# Fig. S25 | Drought-deficit offset

Plots drought-deficit offsets at matched pumping reductions.

In [ ]:
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(p.resolve() for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'management').is_dir())
DATA = ROOT / 'outputs' / 'MANAGEMENT_2012_2013' / 'equal_volume'
OUT = ROOT / 'outputs' / 'figures' / 'FigS25' / 'FigS25.png'
OUT.parent.mkdir(parents=True,exist_ok=True)
data = pd.read_csv(DATA / 'drought_deficit_offset.csv',dtype={'seed':str})
PI75=1.150349
colors={'Uniform':'#4A4A4A','High pumping':'#7489B5','Leverage guided':'#16857C'}
mpl.rcParams.update({'font.family':'Arial','font.size':11,'axes.labelsize':13,'axes.titlesize':12.5,'xtick.labelsize':10.5,'ytick.labelsize':10.5,'axes.linewidth':0.8,'axes.spines.top':True,'axes.spines.right':True})

In [ ]:
def interval(values):
    values=np.asarray(values,float); mean=values.mean(); radius=PI75*values.std(ddof=0)
    return mean,mean-radius,mean+radius

fig,axes=plt.subplots(1,2,figsize=(7.2,3.15),gridspec_kw={'width_ratios':[0.9,1.15]})
strategies=('Uniform','High pumping','Leverage guided')
report=data[data.budget_nominal.eq(20)]
for y,strategy in enumerate(strategies):
    mean,low,high=interval(100*report.loc[report.strategy.eq(strategy),'fraction_offset'])
    axes[0].errorbar(mean,y,xerr=[[mean-low],[high-mean]],fmt='o',color=colors[strategy],ms=5,capsize=3,lw=1.2)
axes[0].set_yticks(range(3),['Uniform','High pumping','Leverage-guided']); axes[0].invert_yaxis()
axes[0].set_xlabel('Deficit offset (%)'); axes[0].set_title('a  Response at 20%',loc='left',fontweight='bold')

for label,control,color,dy in [('Leverage − Uniform','Uniform','#486A9A',-0.12),('Leverage − High pumping','High pumping','#B77A61',0.12)]:
    left=data[data.strategy.eq('Leverage guided')][['budget_nominal','seed','fraction_offset']]
    right=data[data.strategy.eq(control)][['budget_nominal','seed','fraction_offset']]
    paired=left.merge(right,on=['budget_nominal','seed'],suffixes=('_l','_r'))
    paired['difference']=100*(paired.fraction_offset_l-paired.fraction_offset_r)
    for y,budget in enumerate((10,20,30)):
        mean,low,high=interval(paired.loc[paired.budget_nominal.eq(budget),'difference'])
        axes[1].errorbar(mean,y+dy,xerr=[[mean-low],[high-mean]],fmt='o',color=color,ms=5,capsize=3,lw=1.2,label=label if y==0 else None)
axes[1].axvline(0,color='0.4',lw=0.8,ls='--'); axes[1].set_yticks(range(3),['10%','20%','30%']); axes[1].invert_yaxis()
axes[1].set_xlabel('Additional deficit offset (%)'); axes[1].set_ylabel('Pumping-reduction budget'); axes[1].set_title('b  Additional deficit offset',loc='left',fontweight='bold')
handles,names=axes[1].get_legend_handles_labels(); fig.legend(handles,names,frameon=False,fontsize=8.5,loc='upper center',bbox_to_anchor=(0.73,1.03),ncol=2)
for ax in axes:
    ax.grid(axis='x',color='#ECE9E4',lw=0.5)
fig.tight_layout(w_pad=1.1); fig.savefig(OUT,dpi=600,bbox_inches='tight',facecolor='white'); plt.show()